# 01 · Data audit
**Question:** Is the IBM AMLSim sample fit for transaction-monitoring analytics, and what can it NOT support?

The raw archives in `data/raw/amlsim_sample/` are verified against `SHA256SUMS` and extracted unchanged.
Every check below is recomputed live by `src/data_validation.py` (the pipeline writes the same table to
`outputs/tables/data_quality_report.csv`). The pipeline refuses to continue if any check has status `FAIL`.

In [1]:
import sys, json, sqlite3
from pathlib import Path
ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
sys.path.insert(0, str(ROOT))
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from src import config
pd.set_option("display.width", 180); pd.set_option("display.max_columns", 30); pd.set_option("display.precision", 4)
T = lambda name: pd.read_csv(config.TABLES_DIR / f"{name}.csv")
KM = json.loads((config.TABLES_DIR / "key_metrics.json").read_text())
con = sqlite3.connect(config.DB_PATH)
def show(fig_name, width=11):
    img = plt.imread(config.FIGURES_DIR / f"{fig_name}.png")
    h, w = img.shape[:2]
    fig, ax = plt.subplots(figsize=(width, width * h / w)); ax.imshow(img); ax.axis("off")
print("outputs loaded from", config.TABLES_DIR.relative_to(ROOT))

outputs loaded from outputs/tables


In [2]:
from src import ingestion, data_validation
print(ingestion.verify_checksums())
nodes, tx, meta = ingestion.load_raw("combined")
print(meta)
print("nodes:", nodes.shape, "transactions:", tx.shape)
nodes.head()

{'20K_cycle200.tgz': True, '20K_fanin200.tgz': True, '20K_fanin200cycle200.tgz': True}
nodes: 20,000 (1803 fraud nodes)
transactions: 120,558
patterns: 200 cycles and 200 fan-in
nodes: (20000, 4) transactions: (120558, 4)


,nodeid,isFraud,init_balance,fraudStep
0,0,0,184.44,-1
1,1,0,175.80,-1
2,2,0,142.06,-1
3,3,0,125.89,-1
4,4,0,151.13,-1


In [3]:
tx.head()

,sourceNodeId,targetNodeId,value,time
0,216,14730,163.30,1
1,322,5431,143.11,1
2,78,19972,192.33,1
3,248,14820,101.86,1
4,242,18672,113.33,1


In [4]:
report = data_validation.audit(nodes, tx, meta, "combined")
print(report["status"].value_counts().to_string())
report[["section", "check", "value", "status"]]

status
INFO    17
PASS    13
WARN    12


,section,check,value,status
0,schema,nodes columns,"nodeid,isFraud,init_balance,fraudStep",PASS
1,schema,transactions columns,"sourceNodeId,targetNodeId,value,time",PASS
2,schema,nodes dtypes,"nodeid:int64,isFraud:int64,init_balance:float64,fraudStep:int64",INFO
3,schema,transactions dtypes,"sourceNodeId:int64,targetNodeId:int64,value:float64,time:int64",INFO
4,volume,node rows,20000,INFO
5,volume,transaction rows,120558,INFO
6,volume,node rows vs metadata.txt,20000 vs 20000,PASS
7,volume,labelled accounts vs metadata.txt,1804 vs 1803,WARN
8,volume,transaction rows vs metadata.txt,120558 vs 120558,PASS
9,missingness,missing values (nodes),0,PASS


### Warnings that shape the whole project
The notes column explains why each `WARN` is kept rather than 'cleaned away'. The most consequential ones
are generator artifacts (sub-100 amounts and exact duplicates occur only between labelled accounts) and the
absence of any field beyond sender, receiver, amount and a daily step.

In [5]:
report.loc[report.status == "WARN", ["check", "value", "note"]]

,check,value,note
7,labelled accounts vs metadata.txt,1804 vs 1803,metadata.txt differs from nodes.csv; nodes.csv is treated as authoritative
16,accounts with no transactions,20,kept in account table; excluded from behavioural features
17,self-transfers (sender == receiver),15,kept and flagged; excluded from counterparty and network features
19,exact duplicate rows (extra copies),1684,"no transaction id exists, so identical (sender, receiver, amount, step) rows cannot be proven to be errors; they are kept as repeated transfers and flagged"
20,share of duplicated rows between two labelled accounts,1.0,"if ~1.0 the duplicates are a simulator artifact of the typology generator, not organic behaviour"
25,transactions per step (min / median / max),8 / 917 / 1424,"volume ramps up and down over the simulation (start/end effects); use relative, not absolute, velocity"
26,timestamp granularity,"integer simulation step (no clock time, no calendar date)",time-of-day and day-of-week analysis is impossible
31,share of <100 amounts between two labelled accounts,1.0,AMLSim draws background amounts from a range starting at 100; if this share is ~1.0 the <100 band is a generation artifact and must NOT be used as a detection rule
34,labelled share of accounts,0.0902,far higher than any realistic prevalence; precision must be re-expressed at realistic prevalence
38,labelled accounts with fraudStep = -1,906,fraudStep semantics are undocumented and inconsistent with the label; column excluded from all features and from evaluation


In [6]:
# the same audit on the two alternative typology samples (used for robustness)
T("data_quality_report").pivot_table(index="check", columns="dataset", values="value", aggfunc="first").loc[
    ["transaction rows", "labelled (isFraud=1) accounts", "labelled share of accounts",
     "exact duplicate rows (extra copies)", "amounts below 100 (count)", "step max"]]

dataset,combined,cycle,fanin
check,,,
transaction rows,120558,117805,118250
labelled (isFraud=1) accounts,1804,945,954
labelled share of accounts,0.0902,0.0473,0.0477
exact duplicate rows (extra copies),1684,1420,254
amounts below 100 (count),3695,1899,1854
step max,149,149,149


In [7]:
# field availability matrix: what the data can and cannot support
fields = ["sender account", "receiver account", "amount", "step (day index)", "account label (typology member)",
          "initial balance", "timestamp (clock time)", "transaction type", "running balance", "merchant/category",
          "location", "device / IP", "customer demographics / KYC", "payment-fraud label", "loss amount"]
avail = ["yes"] * 6 + ["no"] * 9
pd.DataFrame({"field": fields, "available": avail})

,field,available
0,sender account,yes
1,receiver account,yes
2,amount,yes
3,step (day index),yes
4,account label (typology member),yes
5,initial balance,yes
6,timestamp (clock time),no
7,transaction type,no
8,running balance,no
9,merchant/category,no


In [8]:
data_validation.assert_no_failures(report)
print("AUDIT PASSED: no FAIL checks - pipeline may continue")

AUDIT PASSED: no FAIL checks - pipeline may continue
